<a href="https://colab.research.google.com/github/sathkumara-smna/258739K/blob/main/site_power_ML_cell_to_site.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [57]:
# Fully Data-Driven Telecom Site Power Prediction
# FULLY DATA-DRIVEN ML MODEL, Predicts Cell-wise power and aggregate

# ============================================================
# IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# ============================================================
# LOAD EXCEL FILES FROM GITHUB
# ============================================================

site_db_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/Site%20Database%20from%20Sey.xlsx"
site_power_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/site_power%20from%20Sey.xlsx"
traffic_4g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/4G%20Traffic.xlsx"
traffic_5g_url = "https://raw.githubusercontent.com/sathkumara-smna/258739K/main/5G%20Traffic.xlsx"

# ============================================================
# READ EXCEL FILES
# ============================================================

site_db = pd.read_excel(site_db_url)
site_power = pd.read_excel(site_power_url)
traffic_4g = pd.read_excel(traffic_4g_url)
traffic_5g = pd.read_excel(traffic_5g_url)
site_db.head(2)

,#,Site_ID,Site Name,2G RRUs,3G RRUs,4G RRUs,5G AAUs,2G Boards,3G Boards,4G Boards,5G Boards,BBU 5900,BBU 3900,BBU 3910
0,1,101,AIRPORT_MAHE,3,7,12,3,1,2,2,1,1,1,1
1,2,102,AIRPORT_PRASLIN,2,4,4,0,1,1,1,0,0,1,0


In [58]:
# ============================================================
# RENAME SITE DATABASE COLUMNS
# ============================================================

site_db.columns = [
    '#',
    'Site_ID',
    'Site_Name',
    'RRU_2G',
    'RRU_3G',
    'RRU_4G',
    'AAU_5G',
    'Col_H',
    'Col_I',
    'Boards_4G',
    'Boards_5G',
    'BBU5900',
    'BBU3900',
    'BBU3910'
]
site_db.head(2)

,#,Site_ID,Site_Name,RRU_2G,RRU_3G,RRU_4G,AAU_5G,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910
0,1,101,AIRPORT_MAHE,3,7,12,3,1,2,2,1,1,1,1
1,2,102,AIRPORT_PRASLIN,2,4,4,0,1,1,1,0,0,1,0


In [59]:
# ============================================================
# CREATE TIME FEATURES
# ============================================================

traffic_4g['hour'] = pd.to_datetime(
    traffic_4g['datetime']
).dt.hour

traffic_5g['hour'] = pd.to_datetime(
    traffic_5g['datetime']
).dt.hour
traffic_4g.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour
0,101,10111,11,1,2026-03-01,2026-03-01 00:00,11.90,0
1,101,10111,11,2,2026-03-01,2026-03-01 00:15,12.03,0


In [60]:
# ============================================================
# LTE CELL COUNT PER SITE
# ============================================================

lte_counts = (

    traffic_4g.groupby('Site_ID')['Cell_ID']
    .nunique()
    .to_dict()

)

traffic_4g['lte_cell_count'] = (
    traffic_4g['Site_ID'].map(lte_counts)
)
traffic_4g.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour,lte_cell_count
0,101,10111,11,1,2026-03-01,2026-03-01 00:00,11.90,0,12
1,101,10111,11,2,2026-03-01,2026-03-01 00:15,12.03,0,12


In [61]:
# ============================================================
# NR CELL COUNT PER SITE
# ============================================================

nr_counts = (

    traffic_5g.groupby('Site_ID')['Cell_ID']
    .nunique()
    .to_dict()

)

traffic_5g['nr_cell_count'] = (
    traffic_5g['Site_ID'].map(nr_counts)
)
traffic_5g.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour,nr_cell_count
0,101,1011,1,1,2026-03-01,2026-03-01 00:00,61.20,0,3
1,101,1011,1,2,2026-03-01,2026-03-01 00:15,67.31,0,3


In [62]:
# ============================================================
# MERGE SITE DATABASE TO LTE
# ============================================================

lte_df = traffic_4g.merge(

    site_db,
    on='Site_ID',
    how='left'

)
lte_df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour,lte_cell_count,#,...,RRU_3G,RRU_4G,AAU_5G,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910
0,101,10111,11,1,2026-03-01,2026-03-01 00:00,11.90,0,12,1,...,7,12,3,1,2,2,1,1,1,1
1,101,10111,11,2,2026-03-01,2026-03-01 00:15,12.03,0,12,1,...,7,12,3,1,2,2,1,1,1,1


In [63]:
# ============================================================
# MERGE SITE DATABASE TO NR
# ============================================================

nr_df = traffic_5g.merge(

    site_db,
    on='Site_ID',
    how='left'

)
nr_df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour,nr_cell_count,#,...,RRU_3G,RRU_4G,AAU_5G,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910
0,101,1011,1,1,2026-03-01,2026-03-01 00:00,61.20,0,3,1,...,7,12,3,1,2,2,1,1,1,1
1,101,1011,1,2,2026-03-01,2026-03-01 00:15,67.31,0,3,1,...,7,12,3,1,2,2,1,1,1,1


In [64]:
# ============================================================
# AGGREGATE LTE TRAFFIC
# ============================================================

lte_total = (

    lte_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['traffic_load_mbps']

    .sum()

)

lte_total.rename(
    columns={'traffic_load_mbps': 'total_4g_traffic'},
    inplace=True
)
lte_total.head(2)

,Site_ID,trigger_ID,date,datetime,total_4g_traffic
0,101,1,2026-03-01,2026-03-01 00:00,150.81
1,101,1,2026-03-02,2026-03-02 00:00,154.18


In [65]:
# ============================================================
# AGGREGATE NR TRAFFIC
# ============================================================

nr_total = (

    nr_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['traffic_load_mbps']

    .sum()

)

nr_total.rename(
    columns={'traffic_load_mbps': 'total_5g_traffic'},
    inplace=True
)
nr_total.head(2)

,Site_ID,trigger_ID,date,datetime,total_5g_traffic
0,101,1,2026-03-01,2026-03-01 00:00,172.34
1,101,1,2026-03-02,2026-03-02 00:00,234.09


In [66]:
# ============================================================
# PREPARE LTE TRAINING DATA
# ============================================================

lte_train = lte_df.merge(

    lte_total,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

lte_train = lte_train.merge(

    nr_total,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

lte_train = lte_train.merge(

    site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

lte_train.fillna(0, inplace=True)
lte_train.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour,lte_cell_count,#,...,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910,total_4g_traffic,total_5g_traffic,site_power
0,101,10111,11,1,2026-03-01,2026-03-01 00:00,11.90,0,12,1,...,1,2,2,1,1,1,1,150.81,172.34,6517.6299
1,101,10111,11,2,2026-03-01,2026-03-01 00:15,12.03,0,12,1,...,1,2,2,1,1,1,1,155.48,188.49,6533.5512


In [67]:
# ============================================================
# LTE CELL TARGET POWER
# ============================================================

lte_train['lte_cell_target_power'] = (

    lte_train['site_power'] *

    (
        lte_train['traffic_load_mbps']
        /
        (
            lte_train['total_4g_traffic'] +
            lte_train['total_5g_traffic'] +
            1
        )
    )

)
lte_train.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour,lte_cell_count,#,...,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910,total_4g_traffic,total_5g_traffic,site_power,lte_cell_target_power
0,101,10111,11,1,2026-03-01,2026-03-01 00:00,11.90,0,12,1,...,2,2,1,1,1,1,150.81,172.34,6517.6299,239.271312
1,101,10111,11,2,2026-03-01,2026-03-01 00:15,12.03,0,12,1,...,2,2,1,1,1,1,155.48,188.49,6533.5512,227.841902


In [68]:
# ============================================================
# LTE FEATURES
# ============================================================

lte_features = [

    'traffic_load_mbps',
    'hour',
    'trigger_ID',
    'lte_cell_count',
    'RRU_2G',
    'RRU_3G',
    'RRU_4G',
    'Boards_4G',
    'BBU3900',
    'BBU3910'

]

X_lte = lte_train[lte_features]

y_lte = lte_train['lte_cell_target_power']
print(X_lte.head(2))
print(y_lte.head(2))

   traffic_load_mbps  hour  trigger_ID  lte_cell_count  RRU_2G  RRU_3G  \
0              11.90     0           1              12       3       7   
1              12.03     0           2              12       3       7   

   RRU_4G  Boards_4G  BBU3900  BBU3910  
0      12          2        1        1  
1      12          2        1        1  
0    239.271312
1    227.841902
Name: lte_cell_target_power, dtype: float64


In [69]:
# ============================================================
# LTE TRAIN TEST SPLIT
# ============================================================

X_train_lte, X_test_lte, y_train_lte, y_test_lte = train_test_split(

    X_lte,
    y_lte,
    test_size=0.2,
    random_state=42

)

In [70]:
# ============================================================
# LTE MODEL
# ============================================================

lte_model = RandomForestRegressor(

    n_estimators=100,
    random_state=42

)

lte_model.fit(X_train_lte, y_train_lte)
print("Done")

Done


In [71]:
# ============================================================
# LTE CELL POWER PREDICTION
# ============================================================

lte_df['predicted_lte_cell_power'] = (

    lte_model.predict(
        lte_df[lte_features]
    )

)
lte_df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour,lte_cell_count,#,...,RRU_4G,AAU_5G,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910,predicted_lte_cell_power
0,101,10111,11,1,2026-03-01,2026-03-01 00:00,11.90,0,12,1,...,12,3,1,2,2,1,1,1,1,249.287363
1,101,10111,11,2,2026-03-01,2026-03-01 00:15,12.03,0,12,1,...,12,3,1,2,2,1,1,1,1,234.101790


In [72]:
# ============================================================
# PREPARE NR TRAINING DATA
# ============================================================

nr_train = nr_df.merge(

    lte_total,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

nr_train = nr_train.merge(

    nr_total,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

nr_train = nr_train.merge(

    site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)

nr_train.fillna(0, inplace=True)
nr_train.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour,nr_cell_count,#,...,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910,total_4g_traffic,total_5g_traffic,site_power
0,101,1011,1,1,2026-03-01,2026-03-01 00:00,61.20,0,3,1,...,1,2,2,1,1,1,1,150.81,172.34,6517.6299
1,101,1011,1,2,2026-03-01,2026-03-01 00:15,67.31,0,3,1,...,1,2,2,1,1,1,1,155.48,188.49,6533.5512


In [73]:
# ============================================================
# NR CELL TARGET POWER
# ============================================================

nr_train['nr_cell_target_power'] = (

    nr_train['site_power'] *

    (
        nr_train['traffic_load_mbps']
        /
        (
            nr_train['total_4g_traffic'] +
            nr_train['total_5g_traffic'] +
            1
        )
    )

)
nr_train.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour,nr_cell_count,#,...,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910,total_4g_traffic,total_5g_traffic,site_power,nr_cell_target_power
0,101,1011,1,1,2026-03-01,2026-03-01 00:00,61.20,0,3,1,...,2,2,1,1,1,1,150.81,172.34,6517.6299,1230.538176
1,101,1011,1,2,2026-03-01,2026-03-01 00:15,67.31,0,3,1,...,2,2,1,1,1,1,155.48,188.49,6533.5512,1274.816162


In [74]:
# ============================================================
# NR FEATURES
# ============================================================

nr_features = [

    'traffic_load_mbps',
    'hour',
    'trigger_ID',
    'nr_cell_count',
    'RRU_2G',
    'RRU_3G',
    'AAU_5G',
    'Boards_5G',
    'BBU5900'

]

X_nr = nr_train[nr_features]
y_nr = nr_train['nr_cell_target_power']
print(X_nr.head(2))
print(y_nr.head(2))

   traffic_load_mbps  hour  trigger_ID  nr_cell_count  RRU_2G  RRU_3G  AAU_5G  \
0              61.20     0           1              3       3       7       3   
1              67.31     0           2              3       3       7       3   

   Boards_5G  BBU5900  
0          1        1  
1          1        1  
0    1230.538176
1    1274.816162
Name: nr_cell_target_power, dtype: float64


In [75]:
# ============================================================
# NR TRAIN TEST SPLIT
# ============================================================

X_train_nr, X_test_nr, y_train_nr, y_test_nr = train_test_split(

    X_nr,
    y_nr,
    test_size=0.2,
    random_state=42

)
print(X_train_nr.head(2))
print(y_train_nr.head(2))

       traffic_load_mbps  hour  trigger_ID  nr_cell_count  RRU_2G  RRU_3G  \
21616             157.25     4          17              3       3       7   
23111             773.47    17          72              3       3       6   

       AAU_5G  Boards_5G  BBU5900  
21616       3          1        1  
23111       3          1        1  
21616    2108.556035
23111    1592.978233
Name: nr_cell_target_power, dtype: float64


In [76]:
# ============================================================
# NR MODEL
# ============================================================

nr_model = RandomForestRegressor(

    n_estimators=100,
    random_state=42

)

nr_model.fit(X_train_nr, y_train_nr)
print("NR MODEL Done")

NR MODEL Done


In [77]:
# ============================================================
# NR CELL POWER PREDICTION
# ============================================================

nr_df['predicted_nr_cell_power'] = (

    nr_model.predict(
        nr_df[nr_features]
    )

)
nr_df.head(2)

,Site_ID,Cell_ID,Sector_ID,trigger_ID,date,datetime,traffic_load_mbps,hour,nr_cell_count,#,...,RRU_4G,AAU_5G,Col_H,Col_I,Boards_4G,Boards_5G,BBU5900,BBU3900,BBU3910,predicted_nr_cell_power
0,101,1011,1,1,2026-03-01,2026-03-01 00:00,61.20,0,3,1,...,12,3,1,2,2,1,1,1,1,1261.692123
1,101,1011,1,2,2026-03-01,2026-03-01 00:15,67.31,0,3,1,...,12,3,1,2,2,1,1,1,1,1368.325668


In [78]:
# ============================================================
# LTE SITE POWER
# ============================================================

lte_site_power = (

    lte_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['predicted_lte_cell_power']

    .sum()

)
lte_site_power.head(2)

,Site_ID,trigger_ID,date,datetime,predicted_lte_cell_power
0,101,1,2026-03-01,2026-03-01 00:00,3376.260223
1,101,1,2026-03-02,2026-03-02 00:00,3007.927749


In [79]:
# ============================================================
# NR SITE POWER
# ============================================================

nr_site_power = (

    nr_df.groupby(
        ['Site_ID', 'trigger_ID', 'date', 'datetime'],
        as_index=False
    )['predicted_nr_cell_power']

    .sum()

)
nr_site_power.head(2)

,Site_ID,trigger_ID,date,datetime,predicted_nr_cell_power
0,101,1,2026-03-01,2026-03-01 00:00,3481.094675
1,101,1,2026-03-02,2026-03-02 00:00,4429.331605


In [80]:
# ============================================================
# MERGE LTE + NR SITE POWER
# ============================================================

final_df = lte_site_power.merge(

    nr_site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='outer'

)

final_df.fillna(0, inplace=True)
final_df.head(2)

,Site_ID,trigger_ID,date,datetime,predicted_lte_cell_power,predicted_nr_cell_power
0,101,1,2026-03-01,2026-03-01 00:00,3376.260223,3481.094675
1,101,1,2026-03-02,2026-03-02 00:00,3007.927749,4429.331605


In [81]:
# ============================================================
# FINAL SITE POWER PREDICTION
# ============================================================

final_df['final_predicted_power'] = (

    final_df['predicted_lte_cell_power'] +
    final_df['predicted_nr_cell_power']

)
final_df.head(2)

,Site_ID,trigger_ID,date,datetime,predicted_lte_cell_power,predicted_nr_cell_power,final_predicted_power
0,101,1,2026-03-01,2026-03-01 00:00,3376.260223,3481.094675,6857.354898
1,101,1,2026-03-02,2026-03-02 00:00,3007.927749,4429.331605,7437.259355


In [82]:
# ============================================================
# MERGE ACTUAL SITE POWER
# ============================================================

final_df = final_df.merge(

    site_power,
    on=['Site_ID', 'trigger_ID', 'date', 'datetime'],
    how='left'

)
final_df.head(2)

,Site_ID,trigger_ID,date,datetime,predicted_lte_cell_power,predicted_nr_cell_power,final_predicted_power,site_power
0,101,1,2026-03-01,2026-03-01 00:00,3376.260223,3481.094675,6857.354898,6517.6299
1,101,1,2026-03-02,2026-03-02 00:00,3007.927749,4429.331605,7437.259355,6467.9894


In [83]:
# ============================================================
# MODEL EVALUATION
# ============================================================

mae = mean_absolute_error(

    final_df['site_power'],
    final_df['final_predicted_power']

)

rmse = np.sqrt(

    mean_squared_error(

        final_df['site_power'],
        final_df['final_predicted_power']

    )

)

mape = np.mean(

    np.abs(

        (
            final_df['site_power'] -
            final_df['final_predicted_power']
        )

        /

        final_df['site_power']

    )

) * 100

r2 = r2_score(

    final_df['site_power'],
    final_df['final_predicted_power']

)

# ============================================================
# PRINT RESULTS
# ============================================================

print('================================')
print('FULLY DATA-DRIVEN ML PERFORMANCE')
print('================================')

print(f'MAE  : {round(mae, 2)}')
print(f'RMSE : {round(rmse, 2)}')
print(f'MAPE : {round(mape, 2)} %')
print(f'R2   : {round(r2, 4)}')

FULLY DATA-DRIVEN ML PERFORMANCE
MAE  : 251.78
RMSE : 361.71
MAPE : 6.62 %
R2   : 0.9691


In [84]:
# ============================================================
# ERROR CALCULATION
# ============================================================

final_df['error'] = (

    final_df['site_power'] -
    final_df['final_predicted_power']

)

final_df['error_percentage'] = (

    np.abs(final_df['error'])
    /
    final_df['site_power']

) * 100
final_df.head(2)

,Site_ID,trigger_ID,date,datetime,predicted_lte_cell_power,predicted_nr_cell_power,final_predicted_power,site_power,error,error_percentage
0,101,1,2026-03-01,2026-03-01 00:00,3376.260223,3481.094675,6857.354898,6517.6299,-339.724998,5.212401
1,101,1,2026-03-02,2026-03-02 00:00,3007.927749,4429.331605,7437.259355,6467.9894,-969.269955,14.985645


In [85]:
# ============================================================
# EXPORT RESULTS
# ============================================================

final_df.to_excel(

    'Fully_Data_Driven_Site_Power_Prediction_Cell_to_Site.xlsx',
    index=False

)

print('================================')
print('OUTPUT FILE CREATED')
print('================================')

print('Fully_Data_Driven_Site_Power_Prediction_Cell_to_Site.xlsx')

# ============================================================
# SAMPLE RESULTS
# ============================================================
print("Done")
print(final_df.head(2))



OUTPUT FILE CREATED
Fully_Data_Driven_Site_Power_Prediction_Cell_to_Site.xlsx
Done
   Site_ID  trigger_ID        date          datetime  \
0      101           1  2026-03-01  2026-03-01 00:00   
1      101           1  2026-03-02  2026-03-02 00:00   

   predicted_lte_cell_power  predicted_nr_cell_power  final_predicted_power  \
0               3376.260223              3481.094675            6857.354898   
1               3007.927749              4429.331605            7437.259355   

   site_power       error  error_percentage  
0   6517.6299 -339.724998          5.212401  
1   6467.9894 -969.269955         14.985645  
